# Seeing Machines — Screenshot Companion
**CompSci for Designers 2 — Final Project — Level 3 (The Critic)**

A multimodal RAG system over a personal Steam screenshot archive (470 images, 72 games).
Two independent retrieval routes over one corpus:

- **Level 1:** CLIP joint image–text embeddings — semantic search, exact game filtering, and
  reverse image identification.
- **Level 2:** Gemma 3 4B generates a structured JSON description of every screenshot;
  descriptions are embedded as text, retrieved independently, and a grounded natural-language
  answer is generated on top.

**Reproducibility:** all heavy artifacts (CLIP embeddings, captions, app-id→name mapping) are
cached in the GitHub repo this notebook clones. On a fresh runtime, `Runtime > Run all`
executes top-to-bottom with **no GPU captioning work and no manual uploads** — the captioning
cell sees the cache and skips itself. The GPU is only needed to *load* Gemma for live
grounded answers in the interface.

**Requirements to run:** a HuggingFace account with the (free) Gemma license accepted at
huggingface.co/google/gemma-3-4b-it, and a HF access token for the login cell.
Runtime: T4 GPU (`Runtime > Change runtime type`).

## 0. Setup — clone repo, resolve paths, sanity-check caches

In [1]:
import os

REPO_URL = "https://github.com/Buendiajosemaria/SeeingMachines.git"
REPO_DIR = "/content/SeeingMachines"

if not os.path.isdir(REPO_DIR):
    print("Cloning repo...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    print("Repo already present. Pulling latest...")
    os.system(f"git -C {REPO_DIR} pull --ff-only")

# The repo mirrors the original MyDrive/SeeingMachines/ layout (nested folder).
ROOT_DIR  = f"{REPO_DIR}/SeeingMachines/screenshots"
CACHE_DIR = f"{REPO_DIR}/SeeingMachines/cache"

print("ROOT_DIR: ", ROOT_DIR,  "| exists:", os.path.isdir(ROOT_DIR))
print("CACHE_DIR:", CACHE_DIR, "| exists:", os.path.isdir(CACHE_DIR))

Cloning repo...
ROOT_DIR:  /content/SeeingMachines/SeeingMachines/screenshots | exists: True
CACHE_DIR: /content/SeeingMachines/SeeingMachines/cache | exists: True


In [2]:
!pip install -q open_clip_torch pillow pandas tqdm gradio
!pip install -q -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00


In [3]:
# All imports for the whole notebook live here.
import json
import re
import difflib
from pathlib import Path
!pip install -q sentence-transformers
import numpy as np
import pandas as pd
import torch
import open_clip
import gradio as gr
from PIL import Image
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [4]:
# Cold-start sanity check: every cached artifact this notebook expects.
# If anything is missing, this cell says so loudly BEFORE a later cell
# fails with a confusing traceback.
EXPECTED_CACHE = {
    "appid_to_name.json":          "app id -> game name mapping",
    "clip_embeddings.npy":         "Level 1 CLIP image embeddings",
    "clip_embeddings_index.csv":   "row index for the CLIP embeddings",
    "captions.json":               "Level 2 VLM descriptions (Gemma 3 4B)",
}

missing = []
for fname, desc in EXPECTED_CACHE.items():
    p = Path(CACHE_DIR) / fname
    status = "OK    " if p.exists() else "MISSING"
    print(f"[{status}] {fname:<28} {desc}")
    if not p.exists():
        missing.append(fname)

if missing:
    print(f"\n\u26a0\ufe0f  {len(missing)} cache file(s) missing: {missing}")
    print("The notebook can regenerate them, but that costs GPU time "
          "(captioning ~470 images takes 1.5-2.5h on a T4).")
else:
    print("\nAll caches present \u2014 this run needs no GPU captioning work.")

[OK    ] appid_to_name.json           app id -> game name mapping
[OK    ] clip_embeddings.npy          Level 1 CLIP image embeddings
[OK    ] clip_embeddings_index.csv    row index for the CLIP embeddings
[OK    ] captions.json                Level 2 VLM descriptions (Gemma 3 4B)

All caches present — this run needs no GPU captioning work.


## 1. Scan the corpus
Walks `ROOT/<appid>/...` and records every image under each numeric app-id folder.
The app-id folder name is **ground-truth metadata** (from Steam's own screenshot
folder convention), resolved to a real game name from a static cached mapping —
no live API calls (grader-reproducibility requirement).

In [5]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

def scan_corpus(root_dir: str) -> pd.DataFrame:
    rows = []
    for appid_dir in sorted(Path(root_dir).iterdir()):
        if not appid_dir.is_dir() or not re.fullmatch(r"\d+", appid_dir.name):
            continue
        for f in appid_dir.rglob("*"):
            if f.is_file() and f.suffix.lower() in IMG_EXTS:
                rows.append({"appid": appid_dir.name, "filepath": str(f), "filename": f.name})
    return pd.DataFrame(rows).drop_duplicates(subset="filepath").reset_index(drop=True)

corpus_df = scan_corpus(ROOT_DIR)
print(f"Found {len(corpus_df)} images across {corpus_df['appid'].nunique()} app-id folders.")

# Note: at 470 images this corpus exceeds the brief's 50-150 target for L1/L2
# (300 for L3). Kept deliberately: the size gives the retrieval comparison and
# the (planned) evaluation layer more distractors to work against, and all
# captioning cost has already been paid and cached. See documentation.
corpus_df.head()

Found 470 images across 72 app-id folders.


,appid,filepath,filename
0,10180,/content/SeeingMachines/SeeingMachines/screens...,2014-01-11_00001.jpg
1,10180,/content/SeeingMachines/SeeingMachines/screens...,2014-03-18_00001.jpg
2,10190,/content/SeeingMachines/SeeingMachines/screens...,2014-07-17_00002.jpg
3,105600,/content/SeeingMachines/SeeingMachines/screens...,2015-07-16_00001.jpg
4,105600,/content/SeeingMachines/SeeingMachines/screens...,2013-08-04_00001.jpg


In [6]:
RESOLVED_CACHE = Path(CACHE_DIR) / "appid_to_name.json"

with open(RESOLVED_CACHE) as f:
    appid_to_name = json.load(f)
print(f"Loaded {len(appid_to_name)} app id -> name pairs.")

corpus_df["game_name"] = corpus_df["appid"].map(appid_to_name)

unmapped = corpus_df[corpus_df["game_name"].isna()]
if len(unmapped):
    print(f"\u26a0\ufe0f  {unmapped['appid'].nunique()} appid(s) missing from the mapping: "
          f"{sorted(unmapped['appid'].unique(), key=int)}")

print("\nImages per game:")
print(corpus_df.groupby("game_name").size().sort_values(ascending=False))

Loaded 72 app id -> name pairs.

Images per game:
game_name
METAL GEAR SOLID V: THE PHANTOM PAIN    60
Grand Theft Auto V Legacy               40
Company of Heroes                       29
Tom Clancy's Rainbow Six Siege          22
Garry's Mod                             19
                                        ..
Ring of Elysium                          1
No More Room in Hell                     1
Moonbase Alpha                           1
Team Fortress 2                          1
Tom Clancy's Rainbow Six® Vegas 2        1
Length: 71, dtype: int64


## 2. Level 1 — CLIP joint embedding space
Every image is embedded once with OpenCLIP ViT-B/32 into a joint image–text space;
text queries embed into the same space and cosine similarity ranks the corpus.
Embeddings load from cache; recomputation only happens if the cache is absent.

In [7]:
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")
clip_model = clip_model.to(device).eval()

open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

In [8]:
EMB_PATH = Path(CACHE_DIR) / "clip_embeddings.npy"
EMB_INDEX_PATH = Path(CACHE_DIR) / "clip_embeddings_index.csv"

def embed_corpus(df: pd.DataFrame, force_recompute: bool = False):
    if EMB_PATH.exists() and EMB_INDEX_PATH.exists() and not force_recompute:
        print("Loading cached CLIP embeddings...")
        return np.load(EMB_PATH), pd.read_csv(EMB_INDEX_PATH)
    feats = []
    with torch.no_grad():
        for fp in tqdm(df["filepath"], desc="Embedding images"):
            img = clip_preprocess(Image.open(fp).convert("RGB")).unsqueeze(0).to(device)
            feat = clip_model.encode_image(img)
            feat = feat / feat.norm(dim=-1, keepdim=True)
            feats.append(feat.cpu().numpy()[0])
    feats = np.stack(feats)
    np.save(EMB_PATH, feats)
    df.to_csv(EMB_INDEX_PATH, index=False)
    return feats, df

clip_embeddings, indexed_df = embed_corpus(corpus_df)

# The cached index may carry filepaths from wherever it was originally built
# (e.g. a Drive mount). Rebase them onto the current ROOT_DIR so image loading
# works regardless of where this session cloned the repo.
_old = indexed_df["filepath"].iloc[0]
_old_prefix = _old[:_old.index("/screenshots/") + len("/screenshots")]
if _old_prefix != ROOT_DIR:
    print(f"Rebasing cached filepaths: {_old_prefix} -> {ROOT_DIR}")
    indexed_df["filepath"] = indexed_df["filepath"].str.replace(_old_prefix, ROOT_DIR, n=1, regex=False)

print(f"{len(indexed_df)} images indexed, embedding matrix {clip_embeddings.shape}.")

Loading cached CLIP embeddings...
470 images indexed, embedding matrix (470, 512).


### Level 1 retrieval logic
- `embed_text` + `cosine_topk`: semantic queries ("a nighttime city street").
- `find_named_game` + exact filter: "pull up all screenshots from \<game\>" — a literal
  metadata lookup, since the folder structure already guarantees the answer.
- `embed_query_image`: reverse identification of an unsorted screenshot by
  nearest-neighbor against the indexed corpus, reported with a similarity score.

In [9]:
def embed_text(query: str) -> np.ndarray:
    with torch.no_grad():
        tok = clip_tokenizer([query]).to(device)
        feat = clip_model.encode_text(tok)
        feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.cpu().numpy()[0]

def embed_query_image(pil_img: Image.Image) -> np.ndarray:
    with torch.no_grad():
        img = clip_preprocess(pil_img.convert("RGB")).unsqueeze(0).to(device)
        feat = clip_model.encode_image(img)
        feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.cpu().numpy()[0]

def cosine_topk(query_vec: np.ndarray, top_k: int = 8, min_sim: float = 0.0) -> pd.DataFrame:
    sims = clip_embeddings @ query_vec
    order = np.argsort(-sims)[:top_k]
    results = indexed_df.iloc[order].copy()
    results["similarity"] = sims[order]
    return results[results["similarity"] >= min_sim]

def find_named_game(query: str):
    names = indexed_df["game_name"].dropna().unique().tolist()
    lowered = [n.lower() for n in names]
    match = difflib.get_close_matches(query.lower(), lowered, n=1, cutoff=0.6)
    if match:
        return names[lowered.index(match[0])]
    for n in names:
        if n.lower() in query.lower() or query.lower() in n.lower():
            return n
    return None

def search(query_text: str = None, query_image: Image.Image = None,
           top_k: int = 8, min_sim: float = 0.0):
    """Level 1 search: pure CLIP image-embedding similarity + metadata filtering."""
    if query_image is not None:
        vec = embed_query_image(query_image)
        results = cosine_topk(vec, top_k=top_k, min_sim=min_sim)
        if len(results):
            best = results.iloc[0]
            answer = (f"Best guess: **{best['game_name']}** "
                      f"(similarity {best['similarity']:.2f}, appid {best['appid']})")
        else:
            answer = "No confident match found."
        return answer, results

    if query_text:
        named = find_named_game(query_text)
        wants_all = any(w in query_text.lower() for w in ["all", "every", "pull up", "show me"])
        if named and wants_all:
            results = indexed_df[indexed_df["game_name"] == named].copy()
            results["similarity"] = 1.0
            return f"Showing all {len(results)} screenshots tagged **{named}** (exact metadata match).", results
        vec = embed_text(query_text)
        results = cosine_topk(vec, top_k=top_k, min_sim=min_sim)
        return f'Top {len(results)} semantic matches for: "{query_text}" (Level 1, CLIP)', results

    return "Enter a text query or upload an image.", indexed_df.iloc[0:0]

## 3. Level 2 — Gemma 3 4B: load model
Requires a HuggingFace token with the Gemma license accepted. The model is needed at
demo time for grounded answers, so this loads regardless of caption cache state.

Quantization decisions (all deliberate, all documented failures behind them):
- **4-bit NF4** — a T4 cannot fit the 4B model unquantized.
- **float16 compute** — float32 proved ~2\u00d7 slower during the first captioning run;
  bfloat16 is unsupported on T4 hardware.
- **`do_pan_and_scan=False`** (in the captioning call) — Gemma's default multi-crop
  tiling multiplies visual tokens ~4-6\u00d7 per image for no captioning benefit at
  this resolution.

In [10]:
from huggingface_hub import login
login()

In [11]:
from transformers import AutoProcessor, Gemma3ForConditionalGeneration, BitsAndBytesConfig

VLM_ID = "google/gemma-3-4b-it"

vlm_quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

vlm_model = Gemma3ForConditionalGeneration.from_pretrained(
    VLM_ID,
    device_map="auto",
    quantization_config=vlm_quant_config,
).eval()
vlm_processor = AutoProcessor.from_pretrained(VLM_ID)

print(f"Loaded {VLM_ID} in 4-bit. Memory footprint: {vlm_model.get_memory_footprint() / 1e9:.2f} GB")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


...

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Loaded google/gemma-3-4b-it in 4-bit. Memory footprint: 3.17 GB


## 4. Level 2 — captioning (skips itself when cached)
The description schema is the central design decision of this level:

- `visible_elements` as an explicit object list is what lets Level 2 "see" small things
  a holistic CLIP embedding dilutes away (the "fish in the corner of the frame" problem).
- `guessed_genre_or_title` is the model's **blind** guess — it is never told the real game
  name. Comparing this guess against the App-ID-derived ground truth generates the
  mis-seeing dossier automatically.
- `visible_text` asks for a **short summary, not verbatim transcription** — the original
  verbatim instruction caused 324/470 captions to blow the token budget and truncate on
  text-dense screenshots (scoreboards, HUD readouts). Bounding the field fixed all but one;
  the survivor failed on curly-quote drift instead (see `misseeing_truncation_failures.json`
  and the documentation). Both failure classes are handled below: bounded prompt + quote
  normalization before parsing.

In [12]:
CAPTION_SYSTEM_PROMPT = (
    "You analyze a single video game screenshot. Describe ONLY what is visibly present \u2014 "
    "do not use any outside knowledge of specific game titles. Respond with STRICT JSON only, "
    "no markdown fences, no commentary outside the JSON, matching exactly this schema:\n"
    '{"scene_type": "menu_ui|combat|exploration|cutscene_dialogue|inventory_crafting|map_screen|other", '
    '"visible_elements": ["short noun phrases for distinct objects, creatures, or items you can see"], '
    '"setting_description": "1-2 sentences describing the environment and any action", '
    '"visible_text": "a SHORT summary of on-screen text, max 20 words - do NOT transcribe everything", '
    '"guessed_genre_or_title": "your own blind guess at genre or game title, based only on visuals", '
    '"confidence_notes": "anything you are unsure about"}'
)

def resize_for_vlm(img: Image.Image, max_side: int = 512) -> Image.Image:
    w, h = img.size
    if max(w, h) > max_side:
        scale = max_side / max(w, h)
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    return img

def caption_image(pil_img: Image.Image, max_new_tokens: int = 300) -> dict:
    messages = [
        {"role": "system", "content": [{"type": "text", "text": CAPTION_SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": pil_img},
            {"type": "text", "text": "Describe this screenshot."},
        ]},
    ]
    inputs = vlm_processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
        processor_kwargs={"do_pan_and_scan": False},
    ).to(vlm_model.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        out = vlm_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    raw = vlm_processor.decode(out[0][input_len:], skip_special_tokens=True).strip()

    cleaned = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    cleaned = cleaned.replace("\u201c", '"').replace("\u201d", '"')  # curly-quote drift fix
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"parse_error": True, "raw_text": raw}

In [13]:
CAPTION_CACHE = Path(CACHE_DIR) / "captions.json"
SAVE_EVERY = 10

def load_captions() -> dict:
    if CAPTION_CACHE.exists():
        with open(CAPTION_CACHE) as f:
            return json.load(f)
    return {}

def save_captions(d: dict):
    with open(CAPTION_CACHE, "w") as f:
        json.dump(d, f, indent=2, ensure_ascii=False)

captions = load_captions()
to_caption = [r for r in indexed_df.itertuples() if r.filepath not in captions]
print(f"{len(captions)} cached, {len(to_caption)} left to caption this run.")

for i, row in enumerate(tqdm(to_caption, desc="Captioning")):
    img = resize_for_vlm(Image.open(row.filepath).convert("RGB"))
    captions[row.filepath] = caption_image(img)
    if (i + 1) % SAVE_EVERY == 0 or i == len(to_caption) - 1:
        save_captions(captions)

parse_errors = [fp for fp, v in captions.items() if v.get("parse_error")]
print(f"Caption cache: {len(captions)} total, {len(parse_errors)} known parse failure(s) "
      f"(kept deliberately \u2014 handled via raw-text fallback and documented in the dossier).")

470 cached, 0 left to caption this run.


Captioning: 0it [00:00, ?it/s]

Caption cache: 470 total, 1 known parse failure(s) (kept deliberately — handled via raw-text fallback and documented in the dossier).


## 5. Level 2 — caption retrieval + grounded answers
Captions are flattened to one string per image and embedded with the **same CLIP text
encoder** as Level 1 (one model stack, deliberately). Retrieval over caption embeddings is
fully independent of Level 1's image embeddings — two routes over one corpus.

The grounded answer feeds retrieved captions (text, not images) back into Gemma. The
prompt explicitly asks for **nearest-related results instead of a flat "no match"** —
so "fish" against a fishless corpus surfaces water/underwater scenes with an explanation
rather than a dead end. Passing actual retrieved images into context is Level 3 territory.

In [14]:
from sentence_transformers import SentenceTransformer

def caption_to_text(cap: dict) -> str:
    if cap.get("parse_error"):
        return cap.get("raw_text", "")  # fallback: still searchable, just unstructured
    elements = ", ".join(cap.get("visible_elements", []))
    return (f"{cap.get('setting_description', '')} "
            f"Visible elements: {elements}. "
            f"Scene type: {cap.get('scene_type', '')}. "
            f"On-screen text: {cap.get('visible_text', '')}").strip()

indexed_df["caption_text"] = indexed_df["filepath"].map(lambda fp: caption_to_text(captions.get(fp, {})))
indexed_df["guessed_genre_or_title"] = indexed_df["filepath"].map(
    lambda fp: captions.get(fp, {}).get("guessed_genre_or_title", ""))

# Dedicated sentence-embedding model for caption retrieval. The previous version
# reused CLIP's text encoder here, which caused "hub collapse": its 77-token limit
# truncated long captions, and CLIP's text space isn't trained for text-to-text
# similarity, so a few generic captions matched every query. Documented in the dossier.
st_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

# New cache filename on purpose — the old caption_embeddings.npy is the flawed CLIP version.
CAPTION_EMB_PATH = Path(CACHE_DIR) / "caption_embeddings_st.npy"

def embed_captions(df: pd.DataFrame, force_recompute: bool = False) -> np.ndarray:
    if CAPTION_EMB_PATH.exists() and not force_recompute:
        embs = np.load(CAPTION_EMB_PATH)
        if len(embs) == len(df):
            print("Loading cached caption embeddings (sentence-transformer)...")
            return embs
        print("Cached caption embeddings don't match corpus size — recomputing.")
    texts = df["caption_text"].fillna(" ").replace("", " ").tolist()
    embs = st_model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
    np.save(CAPTION_EMB_PATH, embs)
    return embs

caption_embeddings = embed_captions(indexed_df)

def search_captions(query_text: str, top_k: int = 8, min_sim: float = 0.0) -> pd.DataFrame:
    """Level 2 search: caption similarity via a dedicated sentence-embedding model."""
    vec = st_model.encode([query_text], normalize_embeddings=True)[0]
    sims = caption_embeddings @ vec
    order = np.argsort(-sims)[:top_k]
    results = indexed_df.iloc[order].copy()
    results["similarity"] = sims[order]
    return results[results["similarity"] >= min_sim]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

In [15]:
def generate_grounded_answer(query: str, retrieved_rows: pd.DataFrame, max_new_tokens: int = 200) -> str:
    if len(retrieved_rows) == 0:
        return "No matching screenshots found in the archive for this query."
    context = "\n".join(f"- [{row.game_name}] {row.caption_text}"
                         for row in retrieved_rows.itertuples())
    messages = [
        {"role": "system", "content": [{"type": "text", "text":
            "Answer the question using ONLY the screenshot descriptions provided. "
            "If nothing matches the query directly, do NOT just say there is no match: "
            "instead, point out the most closely related screenshots and explain the connection "
            "(e.g. for 'fish', underwater or water scenes are relevant). Be brief and concrete."}]},
        {"role": "user", "content": [{"type": "text", "text":
            f"Screenshot descriptions:\n{context}\n\nQuestion: {query}"}]},
    ]
    inputs = vlm_processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(vlm_model.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        out = vlm_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return vlm_processor.decode(out[0][input_len:], skip_special_tokens=True).strip()

## 6. Level 1 vs Level 2 — required comparison (≥ shared queries)
Same queries through both routes. "fish" is here on purpose: no dedicated fish screenshots
exist in this corpus, making it the clearest probe of how each route handles absence —
CLIP's holistic image similarity vs. caption vocabulary matching. Interpretation of the
divergence lives in the project documentation.

In [16]:
comparison_queries = [
    "fish",
    "a nighttime city street",
    "a health bar or HUD element",
    "an inventory or crafting menu",
    "a boss fight",
]

for q in comparison_queries:
    print(f"\n=== Query: {q!r} ===")
    l1 = cosine_topk(embed_text(q), top_k=5)
    l2 = search_captions(q, top_k=5)
    print("Level 1 (CLIP image embeddings):")
    print(l1[["game_name", "filename", "similarity"]].to_string(index=False))
    print("Level 2 (caption embeddings):")
    print(l2[["game_name", "filename", "similarity"]].to_string(index=False))


=== Query: 'fish' ===
Level 1 (CLIP image embeddings):
                           game_name             filename  similarity
                              Arma 3 20181223230739_1.jpg    0.220199
                             Besiege 20181117030643_1.jpg    0.217788
METAL GEAR SOLID V: THE PHANTOM PAIN 2015-09-02_00012.jpg    0.209470
                             Besiege 20181117030700_1.jpg    0.209415
         The Elder Scrolls V: Skyrim 2014-07-04_00003.jpg    0.208614
Level 2 (caption embeddings):
                                game_name             filename  similarity
              The Elder Scrolls V: Skyrim 2014-07-04_00003.jpg    0.394415
                              Garry's Mod 2015-08-09_00002.jpg    0.345668
Grand Theft Auto IV: The Complete Edition 2013-12-20_00001.jpg    0.340755
Grand Theft Auto IV: The Complete Edition 2014-07-01_00002.jpg    0.332902
                                   Arma 3 20181223230739_1.jpg    0.315515

=== Query: 'a nighttime city street' ===
Le

## 7. Level 3 — Hybrid retrieval
Fuses the two independent routes into one weighted ranking:
`fused = alpha * clip_sim + (1 - alpha) * caption_sim`, with `alpha` exposed as a live
slider in the interface (Section 11).

**Why the per-query normalization is not optional:** the two similarity scores live on
different scales — CLIP cosines on this corpus cluster around 0.20–0.30, while the MiniLM
caption similarities range up to ~0.6 (visible in the interface screenshots in the
documentation). Fused raw, the caption route silently dominates at *any* alpha, making the
weighting meaningless. Min-max normalizing each route per query puts both on [0, 1] so
alpha actually expresses a preference between routes. This scale mismatch and its fix are
part of the fusion-weighting justification required by the brief.

In [17]:
def normalize(sims: np.ndarray) -> np.ndarray:
    """Min-max per query: CLIP and MiniLM similarities live on different scales,
    so raw fusion would let one route silently dominate at any alpha."""
    lo, hi = sims.min(), sims.max()
    return (sims - lo) / (hi - lo) if hi > lo else np.zeros_like(sims)

def search_hybrid(query_text: str, alpha: float = 0.5, top_k: int = 8) -> pd.DataFrame:
    """Level 3 search. alpha=1.0 -> pure CLIP (Level 1); alpha=0.0 -> pure captions (Level 2)."""
    clip_sims = normalize(clip_embeddings @ embed_text(query_text))
    cap_sims  = normalize(caption_embeddings @ st_model.encode([query_text], normalize_embeddings=True)[0])
    fused = alpha * clip_sims + (1 - alpha) * cap_sims
    order = np.argsort(-fused)[:top_k]
    results = indexed_df.iloc[order].copy()
    results["similarity"] = fused[order]
    results["clip_sim"] = clip_sims[order]
    results["caption_sim"] = cap_sims[order]
    return results

# Quick smoke test: the same query at three alphas.
for a in [1.0, 0.5, 0.0]:
    r = search_hybrid("fish", alpha=a, top_k=3)
    print(f"alpha={a}:", ", ".join(f"{row.game_name} ({row.similarity:.2f})" for row in r.itertuples()))

alpha=1.0: Arma 3 (1.00), Besiege (0.98), METAL GEAR SOLID V: THE PHANTOM PAIN (0.91)
alpha=0.5: The Elder Scrolls V: Skyrim (0.95), Arma 3 (0.90), Besiege (0.88)
alpha=0.0: The Elder Scrolls V: Skyrim (1.00), Garry's Mod (0.87), Grand Theft Auto IV: The Complete Edition (0.86)


## 8. Level 3 — True multimodal answering
At answer time, the retrieved **images themselves** (not their captions) go into Gemma's
context. Capped at 3–4 images: each costs the model ~256 visual tokens, and small VLMs
degrade noticeably as image count rises — per the brief, **documenting that degradation is
a legitimate finding, not a failure**. The probe below generates the raw material: the same
question answered from 1, 2, 3, and 4 images in context.

In [18]:
def generate_multimodal_answer(query: str, retrieved_rows: pd.DataFrame,
                               max_images: int = 3, max_new_tokens: int = 200) -> str:
    """Answer from the retrieved IMAGES themselves, not their captions (Level 3)."""
    rows = retrieved_rows.head(max_images)
    if len(rows) == 0:
        return "No matching screenshots found."
    content = []
    for row in rows.itertuples():
        content.append({"type": "image",
                        "image": resize_for_vlm(Image.open(row.filepath).convert("RGB"))})
    content.append({"type": "text", "text":
        f"These are screenshots retrieved from a personal game archive. Question: {query}\n"
        f"Answer based only on what you can see in these images. Be brief and concrete."})
    messages = [{"role": "user", "content": content}]
    inputs = vlm_processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
        processor_kwargs={"do_pan_and_scan": False},
    ).to(vlm_model.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        out = vlm_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return vlm_processor.decode(out[0][input_len:], skip_special_tokens=True).strip()

In [20]:
# Degradation probe — same query, growing image count. Screenshot this output
# for the documentation: where does the answer get vaguer, start conflating
# images, or miss things it caught with fewer images in context?
probe_query = "what weapons are visible?"
probe_rows = search_hybrid(probe_query, alpha=0.5, top_k=4)
print("Images in context (in retrieval order):")
print(probe_rows[["game_name", "filename"]].to_string(index=False))

for n in [1, 2, 3, 4]:
    print(f"\n=== {n} image(s) in context ===")
    print(generate_multimodal_answer(probe_query, probe_rows, max_images=n))

Images in context (in retrieval order):
                     game_name             filename
                    Far Cry® 4 20170613134138_1.jpg
Totally Accurate Battlegrounds 20180609001952_1.jpg
Tom Clancy's Rainbow Six Siege 20190602202354_1.jpg
         Guns of Icarus Online 2015-07-18_00001.jpg

=== 1 image(s) in context ===


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Based on the images, the visible weapon is a revolver.

=== 2 image(s) in context ===
Here's a breakdown of the visible weapons in the screenshots:

*   **Image 1:** A .44 Magnum revolver
*   **Image 2:** A Mini Gun (a large, automatic weapon)

=== 3 image(s) in context ===
Here's a breakdown of the weapons visible in the screenshots:

*   **Screenshot 1:** A .44 Magnum revolver
*   **Screenshot 2:** A Mini Gun
*   **Screenshot 3:** A breaching torch, M4, and a red assault rifle.

=== 4 image(s) in context ===
Here's a breakdown of the weapons visible in the screenshots:

*   **A4 Magnum Revolver** (first image)
*   **Mini Gun** (second image)
*   **Breaching Torch** (third image)
*   **A5 Bullet Spray** (fourth image)


## 9. Level 3 — Evaluation: gold-standard query set + precision@k
20–30 queries with known-correct images, hand-labeled by the archive's owner (me) — the
one person who can define ground truth for this corpus. Precision@5 is measured for all
three routes: CLIP-only, caption-only, and hybrid at alpha=0.5.

Query design mixes difficulty deliberately: game-specific content, generic visual scenes,
small-object queries (the class Level 2 was built for), and queries predicted to fail —
a predicted failure analyzed well is worth more here than an unexplained success.
The interpretation of *why* the routes differ lives in the evaluation report in the
project documentation; this cell produces the numbers.

**FILL IN `GOLD` before running** — filenames must match `indexed_df["filename"]` exactly.
Use the helper below to look up filenames per game.

In [26]:
# Helper: list filenames for a game to build the gold set.
def files_for(game_substring: str):
    hits = indexed_df[indexed_df["game_name"].str.contains(game_substring, case=False, na=False)]
    return hits[["game_name", "filename"]].to_string(index=False)

# Example: print(files_for("Skyrim"))
GOLD = {
    "lightsaber duel":      ["2014-12-22_00018.jpg"],
    "a sports car":         ["2014-07-10_00002.jpg", "20201218031352_1.jpg"],
    "a group picture":      ["20250625000718_1.jpg"],
    "a game bug":           ["20211219160813_1.jpg", "20260126220957_1.jpg"],
    "a movie reference":    ["20241127022240_1.jpg"],
    "a medieval knight":    ["20250715185021_1.jpg"],
    "a portrait":           ["20201218015311_1.jpg"],
    "a dog":                ["2015-09-02_00014.jpg"],
    "snowy landscape":      ["2015-02-09_00002.jpg"],
    "a futuristic city":    ["20250701233431_1.jpg"],
    "a panoramic view":     ["20201211024812_1.jpg"],
    "golden hour":          ["20240524233558_1.jpg"],
    "balloons":             ["20201214014231_1.jpg"],
    "planet earth":         ["20201109185656_1.jpg"],
    "a scoreboard":         ["20251124020905_1.jpg"],
    "an inventory menu":    ["20250523204812_1.jpg"],
    "a main menu":          ["2015-01-09_00001.jpg"],
    "a management menu":    ["20180208183503_1.jpg"],
    "a fish":               [],   # no fish screenshots exist; expected miss for all routes
    "nudity":               [],   # no NSFW material in the corpus
}

In [29]:
# --- Benchmark 1: game-identity, zero labeling ---
# Query = game name; gold = that game's own files per the App ID folder structure,
# which neither retrieval route ever saw. Only games with >=5 screenshots, so
# precision@5 can reach 1.0 and is a fair metric here (unlike the single-answer set).

def precision_at_k(retrieved: pd.DataFrame, correct: set, k: int = 5) -> float:
    top = retrieved.head(k)["filename"].tolist()
    return sum(1 for f in top if f in correct) / k

game_counts = indexed_df.groupby("game_name").size()
eval_games = [g for g in game_counts[game_counts >= 5].index if not g.startswith("Unknown")]

K = 5
auto_rows = []
for game in eval_games:
    correct = set(indexed_df[indexed_df["game_name"] == game]["filename"])
    auto_rows.append({
        "query": game,
        "n_gold": len(correct),
        "p@5 CLIP":    precision_at_k(cosine_topk(embed_text(game), top_k=K), correct, K),
        "p@5 caption": precision_at_k(search_captions(game, top_k=K), correct, K),
        "p@5 hybrid":  precision_at_k(search_hybrid(game, alpha=0.5, top_k=K), correct, K),
    })

auto_df = pd.DataFrame(auto_rows)
print(auto_df.to_string(index=False))
print(f"\nMean p@5 per route across {len(auto_df)} game-name queries:")
print(auto_df[["p@5 CLIP", "p@5 caption", "p@5 hybrid"]].mean())

auto_df.to_csv(Path(CACHE_DIR) / "evaluation_gameid.csv", index=False)

                                    query  n_gold  p@5 CLIP  p@5 caption  p@5 hybrid
              Arma 2: Operation Arrowhead      14       0.8          0.0         0.4
                                   Arma 3      14       0.2          0.0         0.4
                                  Besiege       5       0.6          0.0         0.6
                        Company of Heroes      29       0.8          0.2         0.6
       Company of Heroes: Opposing Fronts       8       0.0          0.4         0.6
                           Cyberpunk 2077      18       1.0          0.0         0.2
                              DEFCON Demo       8       0.0          0.4         0.2
                                Far Cry 3       6       0.8          0.0         0.4
                               Far Cry® 4      16       0.8          0.0         0.0
                              Garry's Mod      19       0.6          0.2         0.6
Grand Theft Auto IV: The Complete Edition      12       0.8      

In [27]:
def hit_at_k(retrieved: pd.DataFrame, correct: set, k: int = 5):
    """1.0 if any gold image is in the top k, else 0.0. The honest metric
    when most queries have a single correct answer (p@5 caps at 0.2 there)."""
    if not correct:
        return float("nan")
    top = retrieved.head(k)["filename"].tolist()
    return 1.0 if any(f in correct for f in top) else 0.0

K = 5
rows = []
for q, correct_files in GOLD.items():
    correct = set(correct_files)
    rows.append({
        "query": q,
        "n_gold": len(correct),
        "hit@5 CLIP":    hit_at_k(cosine_topk(embed_text(q), top_k=K), correct, K),
        "hit@5 caption": hit_at_k(search_captions(q, top_k=K), correct, K),
        "hit@5 hybrid":  hit_at_k(search_hybrid(q, alpha=0.5, top_k=K), correct, K),
    })

eval_df = pd.DataFrame(rows)
print(eval_df.to_string(index=False))

scored = eval_df.dropna()
print(f"\nHit rate @5 over {len(scored)} queries with gold answers:")
print(scored[["hit@5 CLIP", "hit@5 caption", "hit@5 hybrid"]].mean())

# Where the routes disagree — this is the interpretive material:
only_clip = scored[(scored["hit@5 CLIP"] == 1) & (scored["hit@5 caption"] == 0)]["query"].tolist()
only_cap  = scored[(scored["hit@5 caption"] == 1) & (scored["hit@5 CLIP"] == 0)]["query"].tolist()
both      = scored[(scored["hit@5 CLIP"] == 1) & (scored["hit@5 caption"] == 1)]["query"].tolist()
print("\nOnly CLIP found:   ", only_clip)
print("Only captions found:", only_cap)
print("Both found:         ", both)

eval_df.to_csv(Path(CACHE_DIR) / "evaluation_results.csv", index=False)

            query  n_gold  hit@5 CLIP  hit@5 caption  hit@5 hybrid
  lightsaber duel       1         1.0            1.0           1.0
     a sports car       2         0.0            1.0           0.0
  a group picture       1         0.0            0.0           0.0
       a game bug       2         0.0            0.0           0.0
a movie reference       1         0.0            0.0           0.0
a medieval knight       1         1.0            0.0           1.0
       a portrait       1         0.0            0.0           0.0
            a dog       1         0.0            1.0           1.0
  snowy landscape       1         1.0            0.0           1.0
a futuristic city       1         1.0            1.0           1.0
 a panoramic view       1         0.0            0.0           0.0
      golden hour       1         1.0            0.0           0.0
         balloons       1         0.0            0.0           0.0
     planet earth       1         1.0            1.0          

## 10. Gradio interface
Functional, not polished (per the brief). Five routes: L1, L2, the L1-vs-L2 comparison,
and the two Level 3 modes — hybrid retrieval (with the live fusion slider) and true
multimodal answering. Hybrid gallery captions show the per-route score breakdown so the
slider's effect is legible.

In [28]:
def to_gallery(results, prefix=""):
    return [(r["filepath"], f"{prefix}{r['game_name']} \u00b7 {r.get('similarity', 0):.2f}")
            for _, r in results.iterrows()]

R_L1     = "Level 1 \u2014 CLIP image similarity"
R_L2     = "Level 2 \u2014 caption-grounded"
R_BOTH   = "L1 vs L2 side-by-side"
R_HYBRID = "Level 3 \u2014 hybrid retrieval"
R_MM     = "Level 3 \u2014 multimodal answer"

def gradio_search(query_text, query_image_path, route, alpha):
    if query_image_path:
        pil_img = Image.open(query_image_path).convert("RGB")
        answer, results = search(query_image=pil_img, top_k=12)
        return answer, gr.update(value=to_gallery(results),
                                 label="Reverse identification (Level 1, CLIP)")

    if not query_text:
        return "Enter a text query or upload an image.", gr.update(value=[], label="Results")

    if route == R_L1:
        answer, results = search(query_text=query_text, top_k=12)
        return answer, gr.update(value=to_gallery(results), label=R_L1)

    if route == R_L2:
        results = search_captions(query_text, top_k=8)
        answer = generate_grounded_answer(query_text, results)
        return answer, gr.update(value=to_gallery(results), label="Level 2 \u2014 caption retrieval")

    if route == R_HYBRID:
        results = search_hybrid(query_text, alpha=alpha, top_k=12)
        answer = (f"Hybrid retrieval at alpha={alpha:.2f} "
                  f"(1 = pure CLIP, 0 = pure captions). Per-image breakdown in captions.")
        gallery = [(r["filepath"],
                    f"{r['game_name']} \u00b7 fused {r['similarity']:.2f} "
                    f"(clip {r['clip_sim']:.2f} / cap {r['caption_sim']:.2f})")
                   for _, r in results.iterrows()]
        return answer, gr.update(value=gallery, label=f"Level 3 \u2014 hybrid (\u03b1={alpha:.2f})")

    if route == R_MM:
        results = search_hybrid(query_text, alpha=alpha, top_k=3)
        answer = generate_multimodal_answer(query_text, results, max_images=3)
        return (f"**Level 3 multimodal answer** (3 retrieved images in VLM context, "
                f"\u03b1={alpha:.2f}):\n\n{answer}"), \
               gr.update(value=to_gallery(results), label="Level 3 \u2014 images passed to the VLM")

    # L1 vs L2 side-by-side: one gallery, two columns.
    # Interleaved pairwise so each ROW is a rank: L1's #n on the left, L2's #n on the right.
    l1_results = search(query_text=query_text, top_k=6)[1]
    l2_results = search_captions(query_text, top_k=6)
    grounded = generate_grounded_answer(query_text, l2_results)
    l1_items = to_gallery(l1_results, prefix="[L1] ")
    l2_items = to_gallery(l2_results, prefix="[L2] ")
    interleaved = []
    for a, b in zip(l1_items, l2_items):
        interleaved.extend([a, b])
    return (f"**Level 2 grounded answer:** {grounded}\n\n"
            f"*Gallery below: left column = Level 1 (CLIP), right column = Level 2 (captions), "
            f"aligned by rank.*"), \
           gr.update(value=interleaved, label="L1 (left) vs L2 (right), rank-aligned")

with gr.Blocks(title="Seeing Machines \u2014 Screenshot Companion") as demo:
    gr.Markdown("# Seeing Machines: Screenshot Companion")
    with gr.Row():
        txt = gr.Textbox(label="Ask something", placeholder='e.g. "fish" or "a nighttime city street"')
        img = gr.Image(label="...or upload a screenshot to identify", type="filepath")
    route = gr.Radio(
        [R_L1, R_L2, R_BOTH, R_HYBRID, R_MM],
        value=R_HYBRID, label="Retrieval route",
    )
    alpha = gr.Slider(0.0, 1.0, value=0.5, step=0.05,
                      label="Fusion weight \u03b1 \u2014 1 = pure CLIP (L1), 0 = pure captions (L2). Used by both Level 3 routes.")
    btn = gr.Button("Search")
    answer_box = gr.Markdown()
    gallery = gr.Gallery(label="Results", columns=2, height=600)
    btn.click(gradio_search, inputs=[txt, img, route, alpha], outputs=[answer_box, gallery])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://88c8382a4d9a689c45.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Appendix — maintenance utilities (not part of a normal run)


In [ ]:
# Download all cache artifacts to your machine.
from google.colab import files
for fname in ["captions.json", "clip_embeddings.npy", "clip_embeddings_index.csv",
               "appid_to_name.json", "caption_embeddings.npy",
               "misseeing_truncation_failures.json", "evaluation_results.csv"]:
    p = Path(CACHE_DIR) / fname
    if p.exists():
        files.download(str(p))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>